# Specialist v3 PPO + Heavy-MCTS (`longrun_sparky_value_bc_v2`) — Colab G4 / A100 / L4 一键启动与恢复

当更换电脑或 Colab 实例重置时，在挂载同一 Google Drive 的 GPU 实例（推荐 **RTX PRO 6000 Blackwell G4** 或 **A100 80GB**）中按顺序运行以下单元，即可从 `/content/drive/MyDrive/cr_rl_ppo` 自动恢复代码、数据、最新 `ppo_latest.pt` 与对手池并后台启动训练。

In [ ]:
# 1. 挂载 Google Drive 并恢复 SSD 训练目录 (/content/crrl_training)
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive

DRIVE = Path('/content/drive/MyDrive/cr_rl_ppo')
if not DRIVE.is_dir():
    drive.mount('/content/drive')

RUN_NAME = 'longrun_sparky_value_bc_v2'
ROOT = Path('/content/crrl_training')
PREP = ROOT / 'prepared'
RUN_DIR = ROOT / 'runs' / RUN_NAME
BC_PLAN_B = Path('/content/crrl_sparky_plan_b')
ROOT.mkdir(parents=True, exist_ok=True)
PREP.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 安装缺失依赖
pkgs = {'yaml': 'PyYAML', 'jpype': 'jpype1', 'gymnasium': 'gymnasium', 'pettingzoo': 'pettingzoo', 'tqdm': 'tqdm', 'tensorboard': 'tensorboard', 'zstandard': 'zstandard'}
missing = [v for k, v in pkgs.items() if importlib.util.find_spec(k) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)

# 解压基础模拟器与最新训练代码包
for zpath, needed in [
    (DRIVE / 'preparation/crrl_training_code.zip', not (ROOT / '.cache/vendor/crforge').is_dir()),
    (DRIVE / 'preparation/sparky_value_bc_v2/g4_heavy_mcts_overlay.zip', (DRIVE / 'preparation/sparky_value_bc_v2/g4_heavy_mcts_overlay.zip').is_file()),
]:
    if needed and zpath.is_file():
        local_z = Path('/content') / zpath.name
        shutil.copyfile(zpath, local_z)
        with zipfile.ZipFile(local_z) as zf:
            zf.extractall(ROOT)

tables = Path.home() / '.crforge/tables/16.635.3'
if not tables.is_dir() and (ROOT / 'assets/tables/16.635.3').is_dir():
    tables.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(ROOT / 'assets/tables/16.635.3', tables)

# 解压 Sparky Plan B 数据集
plan_b_zips = sorted((DRIVE / 'preparation/sparky_value_bc_v2').glob('sparky_plan_b_*.zip'))
if plan_b_zips and not (BC_PLAN_B / 'dataset.manifest.json').is_file():
    BC_PLAN_B.mkdir(parents=True, exist_ok=True)
    local_b = Path('/content/sparky_plan_b.zip')
    shutil.copyfile(plan_b_zips[-1], local_b)
    with zipfile.ZipFile(local_b) as zf:
        zf.extractall(BC_PLAN_B)

# 恢复 anchor 与最新一代 checkpoint + 对手池
if not (PREP / 'anchor_actor.pt').is_file():
    shutil.copyfile(DRIVE / 'preparation/anchor_actor.pt', PREP / 'anchor_actor.pt')

sys.path.insert(0, str(ROOT / 'src'))
from cr_rl.training.fast_publication import restore_latest
backup_dir = DRIVE / 'training' / RUN_NAME
if not (RUN_DIR / 'ppo_latest.pt').is_file():
    restored = restore_latest(backup_dir, RUN_DIR)
    print('Restored from Drive backup:', restored)

# 根据当前 GPU 显存自动适配 G4 (95GB) / A100 / L4 配置
import torch, yaml
snap = torch.load(RUN_DIR / 'ppo_latest.pt', map_location='cpu', mmap=True, weights_only=False)
(PREP / 'source_metadata.json').write_text(json.dumps({k: snap.get(k) for k in ('updates', 'model_architecture_version', 'decision_ticks', 'training_args', 'reward_config', 'level_profile')}, indent=2))
(PREP / 'source_league.yaml').write_text(yaml.safe_dump(snap['league_config']))
del snap

cfg_src = DRIVE / 'preparation/sparky_value_bc_v2/cr_train_config_g4.json'
if cfg_src.is_file():
    cfg = json.loads(cfg_src.read_text())
    props = torch.cuda.get_device_properties(0)
    vram_gib = props.total_memory / (1024 ** 3)
    if vram_gib < 45:
        cfg.update(gradient_checkpointing=True, frame_microbatch=256, actor_batch_size=2, critic_batch_size=4, lookahead_distill_parallel_searches=4)
    (PREP / 'cr_train_config.json').write_text(json.dumps(cfg, indent=2))
    print(f'Ready on {props.name} ({vram_gib:.1f} GiB VRAM), run={RUN_NAME}')


In [ ]:
# 2. 启动后台训练守护进程 (cr_train_pipeline.py)
import subprocess, sys
subprocess.run([sys.executable, '/content/crrl_training/scripts/cr_train_pipeline.py', 'start'], check=True)


In [ ]:
# 3. 实时查看训练进度（按中断按钮不会停止后台训练）
import json, time
from pathlib import Path
from IPython.display import clear_output
from cr_rl.training.colab_pipeline import progress

ROOT = Path('/content/crrl_training')
cfg = json.loads((ROOT / 'prepared/cr_train_config.json').read_text())
try:
    while True:
        clear_output(wait=True)
        print(json.dumps(progress(ROOT, cfg['run_name']), indent=2, ensure_ascii=False))
        time.sleep(15)
except KeyboardInterrupt:
    print('已退出面板显示；后台训练仍在继续。')


In [ ]:
# 4. 安全停机（等待当前 checkpoint 保存并同步到 Google Drive 后退出）
# import subprocess, sys
# subprocess.run([sys.executable, '/content/crrl_training/scripts/cr_train_pipeline.py', 'stop'], check=True)
